# TP1 - Regresión e Introducción a la evaluación de modelos

## Dataset: Bike Sharing

El objetivo de este trabajo es predecir la cantidad de bicicletas alquiladas a partir de variables relacionadas con el clima, el calendario y el momento del día.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Carga e inspección del dataset
Usamos el archivo hour.csv del dataset de Bike Sharing

In [ ]:
data = pd.read_csv("../data/hour.csv")
data.head()

## 1. Limpieza de datos

In [ ]:
print("Cantidad de filas:", data.shape[0])
print("Cantidad de columnas:", data.shape[1])

print("\nColumnas:")
print(data.columns.tolist())

print("\nTipos de datos:")
print(data.dtypes)

In [ ]:
data.describe()

El dataset contiene variables relacionadas con fecha, hora, estación del año, clima, temperatura, humedad y cantidad de bicicletas alquiladas.

La variable objetivo que se busca predecir es `cnt`, que representa la cantidad total de bicicletas alquiladas en cada registro horario.

## 1.1 Variables categóricas
## 1.1 Variables categóricas

Aunque varias variables del dataset están almacenadas como números, algunas representan categorías y no cantidades continuas.

En particular:
- `season`: estación del año.
- `yr`: año.
- `mnth`: mes.
- `hr`: hora del día.
- `holiday`: indica si el día es feriado.
- `weekday`: día de la semana.
- `workingday`: indica si es un día laboral.
- `weathersit`: situación climática.

Estas variables se tratarán como categóricas cuando corresponda, ya que sus valores numéricos representan etiquetas y no magnitudes.

In [ ]:
categorical_cols = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit"
]

for col in categorical_cols:
    print(f"\n{col}:")
    print(data[col].value_counts().sort_index())

Para poder utilizar estas variables en los modelos de regresión, se aplicará una codificación adecuada. En las variables categóricas sin orden natural se utilizará One-Hot Encoding, evitando imponer relaciones numéricas artificiales entre las categorías.

## 1.2 Valores faltantes

Se analiza si existen valores faltantes en alguna de las variables del dataset. En caso de encontrarlos, será necesario definir una estrategia de tratamiento antes de entrenar los modelos.

In [ ]:
missing_values = data.isnull().sum()

print("Valores faltantes por columna:")
print(missing_values)

print("\nCantidad total de valores faltantes:")
print(missing_values.sum())

Si el resultado anterior indica que no existen valores faltantes, no será necesario eliminar filas ni aplicar métodos de imputación.

En caso de utilizar una versión del dataset que contenga valores faltantes, se deberá definir una estrategia de tratamiento de acuerdo con la cantidad y el tipo de variable afectada.

## 1.3 Outliers

Se analiza la posible presencia de valores atípicos en las variables numéricas del dataset.

Para detectarlos se utilizará el criterio del rango intercuartílico (IQR). Se consideran potenciales outliers los valores menores que:

Q1 - 1.5 × IQR

o mayores que:

Q3 + 1.5 × IQR

donde IQR = Q3 - Q1.

La detección de un outlier no implica necesariamente su eliminación. En este dataset, valores altos o bajos de variables climáticas o de cantidad de bicicletas alquiladas pueden representar situaciones reales.

In [ ]:
numeric_cols = [
    "temp",
    "atemp",
    "hum",
    "windspeed",
    "casual",
    "registered",
    "cnt"
]

outlier_summary = []

for col in numeric_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    n_outliers = ((data[col] < lower_limit) | 
                  (data[col] > upper_limit)).sum()

    outlier_summary.append([
        col,
        Q1,
        Q3,
        lower_limit,
        upper_limit,
        n_outliers
    ])

outliers_df = pd.DataFrame(
    outlier_summary,
    columns=[
        "Variable",
        "Q1",
        "Q3",
        "Límite inferior",
        "Límite superior",
        "Cantidad de outliers"
    ]
)

outliers_df

for col in numeric_cols:
    plt.figure(figsize=(7, 3))
    plt.boxplot(data[col], vert=False)
    plt.title(f"Boxplot de {col}")
    plt.xlabel(col)
    plt.show()

### Decisión sobre los outliers

Los valores detectados mediante el criterio IQR se consideran inicialmente como posibles valores atípicos, pero no se eliminarán automáticamente.

En el contexto del Bike Sharing Dataset, una cantidad de alquileres especialmente alta, una determinada condición de humedad o valores extremos de viento pueden corresponder a observaciones reales.

Por este motivo, se decidió mantener los datos y evitar eliminar observaciones únicamente por superar los límites establecidos mediante IQR. Esta decisión podrá revisarse posteriormente si se observa que determinados valores afectan significativamente el desempeño de los modelos.

## 1.4 Selección de características

Antes de entrenar los modelos, se debe decidir qué variables utilizar como características de entrada.

La variable objetivo será:

- `cnt`: cantidad total de bicicletas alquiladas.

Se excluirán las siguientes variables:

- `instant`: corresponde únicamente a un identificador de cada registro y no aporta información útil para la predicción.
- `dteday`: representa la fecha. La información temporal relevante ya se encuentra representada mediante variables como año, mes, hora y día de la semana.
- `casual`: cantidad de usuarios casuales.
- `registered`: cantidad de usuarios registrados.

Las variables `casual` y `registered` no se utilizarán porque la variable objetivo `cnt` se obtiene directamente como la suma de ambas. Utilizarlas produciría fuga de información (data leakage), ya que estaríamos utilizando información que determina directamente el valor que queremos predecir.

Se conservarán como variables predictoras las características relacionadas con el calendario, el momento del día y las condiciones climáticas.

In [ ]:
features = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit",
    "temp",
    "atemp",
    "hum",
    "windspeed"
]

target = "cnt"

X = data[features]
y = data[target]

print("Variables de entrada:")
print(X.columns.tolist())

print("\nVariable objetivo:")
print(target)

print("\nDimensiones de X:")
print(X.shape)

print("\nDimensiones de y:")
print(y.shape)

### Escalado de variables

Las variables numéricas continuas presentan diferentes escalas y unidades. Por este motivo, se aplicará estandarización mediante `StandardScaler`.

El escalado se realizará dentro del pipeline de procesamiento y se ajustará únicamente utilizando los datos de entrenamiento. De esta manera se evita que información del conjunto de test influya sobre el entrenamiento del modelo.

Las variables categóricas se transformarán mediante One-Hot Encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

categorical_features = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit"
]

numerical_features = [
    "temp",
    "atemp",
    "hum",
    "windspeed"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

preprocessor

### Conclusión Ejercicio 1

Luego del análisis inicial:

- se identificaron las variables categóricas;
- se analizó la presencia de valores faltantes;
- se evaluaron posibles outliers mediante el criterio IQR;
- se decidió mantener inicialmente los valores atípicos detectados;
- se eliminaron variables que no aportan información útil o que producirían data leakage;
- se definieron las variables categóricas que serán codificadas mediante One-Hot Encoding;
- se definieron las variables numéricas que serán estandarizadas mediante StandardScaler.

Con los datos preparados, el siguiente paso será separar el dataset en conjuntos de entrenamiento y test y comenzar el entrenamiento de los modelos de regresión.